#  Develop an AI RAG app with the Microsoft Foundry SDK

In [1]:
import os
# Here we set the working directory to the project root to ensure imports work correctly
from pathlib import Path
target = "dp100-learn"
p = Path.cwd()
print(f"Starting working directory: {p}")
while p.name != target and p.parent != p:
    p = p.parent
# Set the path to your project root manually if the above code does not work
# p = "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn"
# p = "C:/Users/dmika/DEV/Projects-local/dp100-learn"
os.chdir(p)
print("Changed working directory to:", p)

Starting working directory: c:\Users\dmika\DEV\Projects-local\dp100-learn\tutorials
Changed working directory to: c:\Users\dmika\DEV\Projects-local\dp100-learn


## Install Necessary Packages

In [ ]:
!pip install azure-ai-projects openai

## Load Packages

# Building the App

In [ ]:
from openai import AzureOpenAI
from utils.consts import FOUNDRY_OPEN_AI_ENDPOINT, FOUNDRY_OPEN_AI_KEY, FOUNDRY_CHAT_MODEL, FOUNDRY_EMBEDDING_MODEL, FOUNDRY_SEARCH_URL, FOUNDRY_SEARCH_KEY, FOUNDRY_INDEX_NAME

In [4]:
from openai import AzureOpenAI
from utils.consts import FOUNDRY_OPEN_AI_ENDPOINT, FOUNDRY_OPEN_AI_KEY
chat_client = AzureOpenAI(
    api_version = "2024-12-01-preview",
    azure_endpoint = FOUNDRY_OPEN_AI_ENDPOINT,
    api_key = FOUNDRY_OPEN_AI_KEY
)

In [ ]:
# Initialize prompt with system message
prompt = [
    {"role": "system", "content": "You are a travel assistant that provides information on travel services available from Margie's Travel."}
]

In [8]:
from utils.consts import FOUNDRY_CHAT_MODEL, FOUNDRY_EMBEDDING_MODEL, FOUNDRY_SEARCH_URL, FOUNDRY_SEARCH_KEY, FOUNDRY_INDEX_NAME

# Loop until the user types 'quit'
while True:
    # Get input text
    input_text = input("Enter the prompt (or type 'quit' to exit): ")
    if input_text.lower() == "quit":
        break
    if len(input_text) == 0:
        print("Please enter a prompt.")
        continue

    # Add the user input message to the prompt
    prompt.append({"role": "user", "content": input_text})

    # Additional parameters to apply RAG pattern using the AI Search index
    rag_params = {
        "data_sources": [
            {
                # the following params are used to search the index
                "type": "azure_search",
                "parameters": {
                    "endpoint": FOUNDRY_SEARCH_URL,
                    "index_name": FOUNDRY_INDEX_NAME,
                    "authentication": {
                        "type": "api_key",
                        "key": FOUNDRY_SEARCH_KEY,
                    },
                    # The following params are used to vectorize the query
                    "query_type": "vector",
                    "embedding_dependency": {
                        "type": "deployment_name",
                        "deployment_name": FOUNDRY_EMBEDDING_MODEL,
                    },
                }
            }
        ],
    }

    # Submit the prompt with the data source options and display the response
    response = chat_client.chat.completions.create(
        model=FOUNDRY_CHAT_MODEL,
        messages=prompt,
        extra_body=rag_params
    )
    completion = response.choices[0].message.content
    print(completion)

    # Add the response to the chat history
    prompt.append({"role": "assistant", "content": completion})

The requested information is not found in the retrieved data. Please try another query or topic.


In [ ]:
import os
from dotenv import load_dotenv
from openai import AzureOpenAI

def main():
    # Clear the console
    os.system('cls' if os.name == 'nt' else 'clear')

    try:
        # Get configuration settings
        load_dotenv()
        open_ai_endpoint = os.getenv("OPEN_AI_ENDPOINT")
        open_ai_key = os.getenv("OPEN_AI_KEY")
        chat_model = os.getenv("CHAT_MODEL")
        embedding_model = os.getenv("EMBEDDING_MODEL")
        search_url = os.getenv("SEARCH_ENDPOINT")
        search_key = os.getenv("SEARCH_KEY")
        index_name = os.getenv("INDEX_NAME")


        # Get an Azure OpenAI chat client
        chat_client = AzureOpenAI(
            api_version = "2024-12-01-preview",
            azure_endpoint = open_ai_endpoint,
            api_key = open_ai_key
        )


        # Initialize prompt with system message
        prompt = [
            {"role": "system", "content": "You are a travel assistant that provides information on travel services available from Margie's Travel."}
        ]

        # Loop until the user types 'quit'
        while True:
            # Get input text
            input_text = input("Enter the prompt (or type 'quit' to exit): ")
            if input_text.lower() == "quit":
                break
            if len(input_text) == 0:
                print("Please enter a prompt.")
                continue

            # Add the user input message to the prompt
            prompt.append({"role": "user", "content": input_text})

            # Additional parameters to apply RAG pattern using the AI Search index
            rag_params = {
                "data_sources": [
                    {
                        # he following params are used to search the index
                        "type": "azure_search",
                        "parameters": {
                            "endpoint": search_url,
                            "index_name": index_name,
                            "authentication": {
                                "type": "api_key",
                                "key": search_key,
                            },
                            # The following params are used to vectorize the query
                            "query_type": "vector",
                            "embedding_dependency": {
                                "type": "deployment_name",
                                "deployment_name": embedding_model,
                            },
                        }
                    }
                ],
            }

            # Submit the prompt with the data source options and display the response
            response = chat_client.chat.completions.create(
                model=chat_model,
                messages=prompt,
                extra_body=rag_params
            )
            completion = response.choices[0].message.content
            print(completion)

            # Add the response to the chat history
            prompt.append({"role": "assistant", "content": completion})

    except Exception as ex:
        print(ex)

if __name__ == '__main__':
    main()